# CO2SITE — Task A | Notebook A: Fictional Learning Example

> **How to derive scientifically justified indifference (q) and preference (p) thresholds for CO₂ storage capacity in PROMETHEE.**
>
> We aim to establish a reproducible procedure that can be applied to multiple CO₂ storage sites and assess whether it is sufficiently robust for implementation in the CO2SITE platform.

**A visual learning notebook — fictional data only.** All statistical figures are generated in Matplotlib directly beside the corresponding explanation. All 16 figures and seven numbered tables use a consistent terminology. This notebook teaches concepts; it **does not** establish a validated indifference threshold for CO2SITE.

**How to use:** select **Runtime → Run all**. The source code is collapsed by default; click **Show code** only when you want to inspect it. Read each explanation, its figure, and the interpretation before moving on.

**Learning sequence:** 1. Terminology → 2. Five hand-calculated pairs → 3. Monte Carlo site capacities → 4. Uncertainty in the difference → 5. Exploratory q and illustrative p → 6. Normal versus PERT → 7. Transfer to GEUS.

For research cases using published geological inputs, open [Notebook B — GEUS-Based Cases](https://colab.research.google.com/github/AnaSoles/ggg-co2-storage-eval/blob/research/task-a-capacity-indifference/examples/colab_task_a_capacity_q.ipynb).

## 1. Common terminology — use these names in both notebooks

| Standard term | Symbol | Definition |
|---|---|---|
| **Site A (reference site)** | A | First site in a comparison |
| **Site B (comparison site)** | B | Second site in a comparison |
| **Simulated capacity** | $C_{A,i}$, $C_{B,i}$ | Possible storage-capacity value for each site in draw $i$ (Mt CO₂) |
| **Draw / Monte Carlo realisation** | $i$ | Number identifying a simulated value/scenario |
| **Paired comparison** | $(C_{A,i},C_{B,i})$ | Two values selected for one subtraction |
| **Capacity difference** | $\Delta_i=C_{B,i}-C_{A,i}$ | Positive differences favour B *on capacity alone* |
| **Mean** | $\mu$ | Average of a distribution |
| **Standard deviation (SD)** | $\sigma$ | Spread of values around the mean |
| **Variance** | $\sigma^2$ | Average squared deviation from the mean |
| **Statistical quantile** | $Q_{10},Q_{50},Q_{90}$ | Values with 10%, 50%, 90% of model outcomes below them |
| **Exceedance capacity** | P90, P50, P10 | Respectively $Q_{10},Q_{50},Q_{90}$ for a continuous capacity distribution |
| **Indifference threshold** | $q$ | PROMETHEE difference regarded as negligible by the decision rule |
| **Preference threshold** | $p$ | PROMETHEE difference reaching full preference |

**Important: lower-case $q$ is NOT upper-case $Q$.** $Q_{10}$ is a statistical quantile; $q$ is a PROMETHEE parameter.

**Paired draws versus a joint uncertainty model:** pairing two values for subtraction does *not* establish correlation. A joint uncertainty model explicitly describes dependence between random inputs or sites. In Section 2 we invent five illustrative pairs; in Section 3 we generate independent normal capacities and pair draws by index; Notebook B independently resamples previously generated capacities. These are distinct procedures.

Both notebooks express capacity in **Mt CO₂** and define the signed difference as **B − A**.

### Figure 1 — From two simulated capacities to one capacity difference

A paired comparison contains one possible capacity for each site. We subtract Site A from Site B. For the first fictional draw, B = 110 Mt and A = 95 Mt, so the difference is +15 Mt.

In [ ]:
# @title Figure 1 — A single paired capacity comparison
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 125, "font.size": 10, "axes.titlesize": 13})
BLUE, ORANGE, RED, GREEN = "#2563EB", "#E49B35", "#BB4852", "#187C70"

fig, ax = plt.subplots(figsize=(10, 3.7))
fig.subplots_adjust(left=.15, right=.96, bottom=.25, top=.83)
ax.barh([1,0], [95,110], height=.36, color=[BLUE,ORANGE], alpha=.8)
for yi, value in [(1,95),(0,110)]:
    ax.text(value+2,yi,f"{value} Mt",va="center",weight="bold")
ax.annotate("",xy=(110,.5),xytext=(95,.5),
            arrowprops=dict(arrowstyle="<->",lw=2,color=GREEN))
ax.text(102.5,.65,"Δ = 110 − 95 = +15 Mt",ha="center",color=GREEN,weight="bold")
ax.set(yticks=[1,0],yticklabels=["Site A","Site B"],xlim=(0,133),ylim=(-.5,1.7),
       xlabel="Simulated static storage capacity (Mt CO₂)",
       title="Figure 1 — A paired comparison produces a single difference")
ax.spines[["top","right","left"]].set_visible(False)
ax.grid(axis="x",alpha=.12)
plt.show()

**Interpretation:** A +15 Mt difference means that B has greater capacity in this *one* pair. It does not mean that B always has greater capacity, nor that the PROMETHEE preference degree is 15.

---

## 2. Five fictional paired comparisons — calculate everything by hand

### 2.1 Where do these values come from?

**Table 1 contains invented teaching values.** They are not GEUS results and are not taken from the larger Monte Carlo model in Section 3. Each row is a paired comparison.

**Table 1 — Five fictional capacity pairs and their differences**

| Draw $i$ | $C_{A,i}$ (Mt) | $C_{B,i}$ (Mt) | $\Delta_i=B-A$ (Mt) |
|---|---:|---:|---:|
| 1 | 95 | 110 | **+15** |
| 2 | 108 | 102 | **−6** |
| 3 | 100 | 120 | **+20** |
| 4 | 112 | 105 | **−7** |
| 5 | 90 | 115 | **+25** |

<details><summary>Show capacity-difference formula and definitions</summary>

$$\Delta_i=C_{B,i}-C_{A,i}$$

$C_{A,i}$ and $C_{B,i}$ are simulated capacities of A and B in draw $i$. $\Delta_i$ is the signed difference in Mt CO₂; $i$ is the draw number.

For draw 2: $\Delta_2=102-108=-6$ Mt, meaning A has more capacity in that pair.

</details>

### Figure 2 — Storage capacities of five fictional paired draws

The grouped bars show the two original capacities in each row of Table 1, before subtraction.

In [ ]:
# @title Table 1 check and Figure 2 — Five paired capacities
a_small = np.array([95,108,100,112,90],dtype=float)
b_small = np.array([110,102,120,105,115],dtype=float)
diff_small = b_small-a_small
print("Table 1 — Verify the five invented paired comparisons")
display(pd.DataFrame({
    "Draw i":np.arange(1,6), "Site A (Mt)":a_small,
    "Site B (Mt)":b_small,"Difference B − A (Mt)":diff_small
}))
i=np.arange(1,6)
fig,ax=plt.subplots(figsize=(10,4.7))
fig.subplots_adjust(left=.09,right=.97,bottom=.18,top=.83)
ax.bar(i-.18,a_small,width=.36,color=BLUE,label="Site A")
ax.bar(i+.18,b_small,width=.36,color=ORANGE,label="Site B")
ax.set(xlim=(.35,5.65),ylim=(0,142),xticks=i,xlabel="Paired draw i",
       ylabel="Simulated capacity (Mt CO₂)",
       title="Figure 2 — Site A and Site B capacities in five fictional draws")
ax.legend(ncol=2,frameon=False,loc="upper right")
for values,offset in [(a_small,-.18),(b_small,.18)]:
    for j,val in enumerate(values):
        ax.text(i[j]+offset,val+2,f"{val:.0f}",ha="center",fontsize=9)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y",alpha=.12)
plt.show()

**Interpretation:** Site B has greater capacity in draws 1, 3 and 5, while Site A is greater in draws 2 and 4. The difference must be calculated for each pair; comparing the averages alone would hide these reversals.

### 2.2 Why is $P(B>A)=60\%$ in this classroom example?

We count the number of *positive differences* in Table 1: three of the five. This proportion describes the five paired values we invented, **not** a scientifically stable probability.

<details><summary>Show the probability calculation</summary>

$$\widehat P(B>A)=\frac{\#\{\Delta_i>0\}}{N}=\frac{3}{5}=0.60=60\%$$

$N$ is the number of pairs; $\#$ counts the positive differences.

</details>

### Figure 3 — Positive and negative capacity differences

Each bar is one value in the final column of Table 1. Bars above zero favour B; bars below zero favour A. The plot helps verify why the observed proportion is 60%.

In [ ]:
# @title Figure 3 — Five signed differences
positive = int(np.sum(diff_small>0))
print(f"Positive differences = {positive}/{len(diff_small)} = {positive/len(diff_small):.0%}")
fig,ax=plt.subplots(figsize=(10,4.8))
fig.subplots_adjust(left=.09,right=.97,bottom=.18,top=.83)
colours=[GREEN if v>0 else RED for v in diff_small]
ax.bar(i,diff_small,color=colours,width=.66,alpha=.85)
ax.axhline(0,color="black",lw=1)
for k,v in enumerate(diff_small):
    ax.annotate(f"{v:+.0f} Mt",(i[k],v),xytext=(0,6 if v>=0 else -17),
                textcoords="offset points",ha="center",fontsize=10)
ax.set(xlim=(.4,5.6),ylim=(-15,34),xticks=i,xlabel="Paired draw i",
       ylabel="Signed difference B − A (Mt CO₂)",
       title="Figure 3 — Three positive and two negative differences")
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y",alpha=.12)
plt.show()

**Interpretation:** The bar height expresses a difference in Mt, **not** its probability. This very small teaching dataset is not sufficient to derive q.

---

## 3. New fictional Monte Carlo experiment — 20,000 draws

**This is a separate fictional experiment, not a continuation of the five fixed values above.**

### 3.1 Model assumptions: mean, SD, and distribution type

**Table 2 — Fictional normal-model inputs, not simulated outputs**

| Input | Site A | Site B | Meaning |
|---|---:|---:|---|
| Assumed mean capacity | 100 Mt | 112 Mt | Centre of each distribution |
| Assumed SD | 8 Mt | 9 Mt | Spread of possible capacities |
| Distribution | Normal | Normal | Teaching assumption, not a geological claim |
| Number of draws | 20,000 | 20,000 | Same sample size for both sites |
| Random generator seed | 2026 | 2026 (one generator) | Reproducible *independent* consecutive draws |

A normal model is written as $N(\mu,\sigma^2)$. The second argument is the **variance**, not the SD: for Site A, $8^2=64$ Mt².

### Figure 4 — Site A normal distribution, mean and ±1 SD

For Site A, the central 68.3% interval is 100 ± 8 = **92–108 Mt**. The remaining 31.7% is divided between the two tails (about 15.9% each).

In [ ]:
# @title Figure 4 — Site A normal distribution with 68.3% coverage
def normal_pdf(v,mean,sd):
    return np.exp(-.5*((v-mean)/sd)**2)/(sd*np.sqrt(2*np.pi))

def normal_68_plot(mean,sd,name,color):
    xx=np.linspace(mean-4.2*sd,mean+4.2*sd,1400)
    yy=normal_pdf(xx,mean,sd)
    low,high=mean-sd,mean+sd
    fig,ax=plt.subplots(figsize=(10,4.8))
    fig.subplots_adjust(left=.09,right=.97,bottom=.19,top=.84)
    ax.plot(xx,yy,color=color,lw=2.8)
    ax.fill_between(xx,yy,where=(xx>=low)&(xx<=high),color="#B9D6FA",alpha=.85)
    ax.fill_between(xx,yy,where=(xx<low)|(xx>high),color="#F0B7BA",alpha=.72)
    for v in (low,mean,high):
        ax.axvline(v,color=color,lw=1.3,ls="--")
    ax.text(mean,yy.max()*.55,"68.3%",color="#184C96",weight="bold",ha="center",fontsize=17)
    ax.text(mean-2.7*sd,yy.max()*.24,"15.9%",ha="center",color="#A13F49",fontsize=11)
    ax.text(mean+2.7*sd,yy.max()*.24,"15.9%",ha="center",color="#A13F49",fontsize=11)
    ax.set(xlim=(mean-4.2*sd,mean+4.2*sd),ylim=(0,yy.max()*1.2),
           xlabel="Simulated capacity (Mt CO₂)",
           title=f"Figure {4 if name=='A' else 5} — Fictional Site {name}: mean ±1 SD")
    ax.set_xticks([mean-2*sd,low,mean,high,mean+2*sd])
    ax.set_xticklabels([f"{mean-2*sd:.0f}",f"{low:.0f}\nμ−σ",
                        f"{mean:.0f}\nmean μ",f"{high:.0f}\nμ+σ",f"{mean+2*sd:.0f}"])
    ax.set_yticks([])
    ax.spines[["left","right","top"]].set_visible(False)
    plt.show()

normal_68_plot(100,8,"A",BLUE)

**Interpretation:** A simulated capacity of 115 Mt would be outside the 92–108 Mt central interval but is **not automatically an error**. The tails represent less frequent outcomes under the assumed normal model.

### Figure 5 — Site B normal distribution, mean and ±1 SD

Site B has a mean of 112 Mt and SD of 9 Mt. Therefore, approximately 68.3% of its capacities are between **103 and 121 Mt**.

In [ ]:
# @title Figure 5 — Site B normal distribution with 68.3% coverage
normal_68_plot(112,9,"B",ORANGE)

**Interpretation:** Site B has a higher assumed mean capacity, but its distribution is also dispersed. The distributions may overlap, so a draw for Site A can exceed a draw for Site B.

### 3.2 Generate 20,000 independent normal capacities

We now generate **new** independent random capacities for both sites using Table 2. We calculate $\Delta_i=B_i-A_i$ in every iteration.

**Table 3** lists the first ten draws for traceability. **Table 4** compares assumed input parameters with the *sample statistics* produced by Monte Carlo.

In [ ]:
# @title Tables 3 and 4 — Simulate and summarise 20,000 draws
N=20_000
rng=np.random.default_rng(2026)
a=rng.normal(loc=100,scale=8,size=N)
b=rng.normal(loc=112,scale=9,size=N)
difference=b-a
print("Table 3 — First ten of 20,000 simulated paired comparisons")
display(pd.DataFrame({
    "Draw i":np.arange(1,11),
    "A capacity (Mt)":a[:10],"B capacity (Mt)":b[:10],
    "Difference B − A (Mt)":difference[:10]
}).round(2))
print("Table 4 — Assumed vs simulated individual-capacity statistics")
display(pd.DataFrame({
    "Site":["A","B"],
    "Assumed mean (Mt)":[100,112],
    "Simulated mean (Mt)":[a.mean(),b.mean()],
    "Assumed SD (Mt)":[8,9],
    "Simulated SD (Mt)":[a.std(ddof=1),b.std(ddof=1)],
    "Number of draws":[N,N]
}).round(2))

### Figure 6 — Monte Carlo capacity distributions for both sites

The histograms show the generated capacities that were summarised in Table 4. **Dashed lines mark simulated means**; the legend reports simulated SDs. The curves are not the five fixed draws of Section 2.

In [ ]:
# @title Figure 6 — Capacity histograms with simulated means and SDs
fig,ax=plt.subplots(figsize=(10.5,5.2))
fig.subplots_adjust(left=.09,right=.97,bottom=.17,top=.85)
for values,name,col in [(a,"Site A",BLUE),(b,"Site B",ORANGE)]:
    mean,sd=values.mean(),values.std(ddof=1)
    ax.hist(values,bins=66,density=True,alpha=.28,color=col,
            label=f"{name}: mean={mean:.2f} Mt; SD={sd:.2f} Mt")
    ax.axvline(mean,color=col,ls="--",lw=1.9)
    ax.plot(np.linspace(mean-3*sd,mean+3*sd,250),
            normal_pdf(np.linspace(mean-3*sd,mean+3*sd,250),mean,sd),
            lw=1.7,color=col,alpha=.85)
ax.set(xlabel="Simulated capacity (Mt CO₂)",ylabel="Probability density",
       title="Figure 6 — 20,000 simulated capacities per site")
ax.legend(loc="upper right",frameon=True,fontsize=9)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y",alpha=.12)
plt.show()

**Interpretation:** The difference between a simulated mean and the assumed mean is sampling variation. Even when B has higher mean capacity, both density curves overlap.

---

## 4. Compare capacities statistically — uncertainty of the difference

### 4.1 Why are individual SD and difference SD different?

From Table 2, Site A has $\sigma_A=8$ Mt and Site B has $\sigma_B=9$ Mt. Under **independence**:

<details><summary>Show formula for the SD of the difference</summary>

$$\sigma_{\Delta,\mathrm{theory}}=\sqrt{\sigma_A^2+\sigma_B^2}=\sqrt{8^2+9^2}=\sqrt{145}\approx12.04\ \mathrm{Mt}.$$

With dependent uncertainties, the variance of the difference includes a covariance term. A shared seed alone does **not** establish dependence or independence.

</details>

### Figure 7 — Comparing the three standard deviations

The first two SDs describe uncertainty of **individual capacities**. The third describes uncertainty in **the advantage B − A**. It is not obtained by subtracting SDs.

In [ ]:
# @title Figure 7 — Individual and difference standard deviations
sd_theory=float(np.sqrt(8**2+9**2))
sd_sim=float(difference.std(ddof=1))
fig,ax=plt.subplots(figsize=(10,4.3))
fig.subplots_adjust(left=.28,right=.96,bottom=.2,top=.84)
names=["Site A capacity","Site B capacity","Difference B − A"]
values=[8,9,sd_theory]
bars=ax.barh(names,values,color=[BLUE,ORANGE,GREEN],height=.52,alpha=.83)
for bar,v in zip(bars,values):
    ax.text(v+.35,bar.get_y()+bar.get_height()/2,f"{v:.2f} Mt",va="center",weight="bold")
ax.set(xlim=(0,16),xlabel="Standard deviation (Mt CO₂)",
       title="Figure 7 — Spread of each site's capacity vs spread of B − A")
ax.invert_yaxis()
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="x",alpha=.13)
plt.show()
print(f"Theoretical SD of B−A: {sd_theory:.2f} Mt; simulated SD: {sd_sim:.2f} Mt")

**Interpretation:** Theoretical $\sigma_\Delta\approx12.04$ Mt and simulated SD $\approx12.00$ Mt describe the **same conceptual quantity**. Small differences arise from finite random sampling. The theoretical mean difference is 112 − 100 = 12 Mt.

### 4.2 How likely is Site B to have more capacity?

For all 20,000 draws, we calculate the proportion with a positive difference. The **vertical zero line** separates outcomes favouring A from outcomes favouring B.

**Table 5 — Statistical summary of capacity differences** includes the difference mean, theoretical and simulated SD, the observed probability of B>A, and the **central 95% model-based uncertainty interval for possible differences**.

<details><summary>Show probability and quantile definitions</summary>

$$\widehat{P}(B>A)=\frac{1}{N}\sum_{i=1}^{N}\mathbf{1}(\Delta_i>0)$$

The indicator $\mathbf{1}$ equals 1 when the difference is positive and 0 otherwise.

$$[Q_{2.5\%}(\Delta),Q_{97.5\%}(\Delta)]$$

This interval includes the central 95% of **simulated possible differences**. It is *not* a frequentist 95% confidence interval for the estimated mean.

</details>

In [ ]:
# @title Table 5 — Difference statistics
lower,upper=np.quantile(difference,[.025,.975])
p_b=float(np.mean(difference>0))
sd_sim=float(difference.std(ddof=1))
print("Table 5 — Statistics of the simulated B − A distribution")
display(pd.DataFrame({
    "Statistic":[
        "Theoretical mean difference B − A",
        "Simulated mean difference B − A",
        "Theoretical SD of difference (independent)",
        "Simulated SD of difference",
        "P(B > A) from simulation",
        "Statistical Q2.5 of differences",
        "Statistical Q97.5 of differences"],
    "Value":[
        "12.00 Mt",
        f"{difference.mean():.2f} Mt",
        f"{sd_theory:.2f} Mt",
        f"{sd_sim:.2f} Mt",
        f"{p_b:.1%}",
        f"{lower:.2f} Mt",
        f"{upper:.2f} Mt"]
}))

### Figure 8 — How often does Site B exceed Site A?

The two coloured sides of the histogram contain the same simulated differences from Table 5. Red bars have $\Delta<0$ (A exceeds B); blue bars have $\Delta>0$ (B exceeds A).

In [ ]:
# @title Figure 8 — Probability B > A in the difference histogram
diff_bins=np.linspace(-38,62,81)
counts,edges=np.histogram(difference,bins=diff_bins)
centres=(edges[:-1]+edges[1:])/2
fig,ax=plt.subplots(figsize=(10.5,5))
fig.subplots_adjust(left=.09,right=.97,bottom=.18,top=.85)
for val,xmid,width in zip(counts,centres,np.diff(edges)):
    ax.bar(xmid,val,width=width,color="#A9CCF4" if xmid>=0 else "#EFB8BD",
           edgecolor="white",linewidth=.25)
ax.axvline(0,color=RED,lw=2,ls="--")
ax.axvline(difference.mean(),color=BLUE,lw=2)
ax.text(.03,.95,f"P(A > B) ≈ {1-p_b:.1%}",transform=ax.transAxes,
        color="#94323D",va="top",fontsize=12,weight="bold")
ax.text(.72,.95,f"P(B > A) ≈ {p_b:.1%}",transform=ax.transAxes,
        color="#154A90",va="top",fontsize=12,weight="bold")
ax.set(xlabel="Capacity difference Δ = B − A (Mt CO₂)",
       ylabel="Number of draws per bin",
       title="Figure 8 — Positive vs negative Monte Carlo differences")
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y",alpha=.12)
plt.show()

**Interpretation:** $P(B>A)\approx84\%$ describes how often B has more simulated capacity than A. It is **not** the PROMETHEE preference degree and **not** a site-suitability probability.

### Figure 9 — Central 95% interval of possible differences

The statistical quantiles $Q_{2.5}$ and $Q_{97.5}$ divide the 20,000 differences so that about 2.5% fall below the lower bound, 95% lie between the bounds, and 2.5% lie above the upper bound.

In [ ]:
# @title Figure 9 — Central 95% interval in the difference histogram
fig,ax=plt.subplots(figsize=(10.5,5.1))
fig.subplots_adjust(left=.09,right=.97,bottom=.19,top=.85)
for val,xmid,width in zip(counts,centres,np.diff(edges)):
    ax.bar(xmid,val,width=width,color="#A9CCF4" if lower<=xmid<=upper else "#EFB8BD",
           edgecolor="white",linewidth=.25)
for value in (lower,upper):
    ax.axvline(value,color=RED,lw=1.9,ls="--")
ax.axvline(0,color="#646D7C",lw=1.3,ls=":")
ax.axvline(difference.mean(),color=BLUE,lw=1.7)
ax.text(.51,.93,"95% central interval",transform=ax.transAxes,
        ha="center",va="top",color="#154A90",fontsize=12,weight="bold")
ax.text(.08,.49,"2.5%",transform=ax.transAxes,color="#A33A44",fontsize=12)
ax.text(.90,.49,"2.5%",transform=ax.transAxes,color="#A33A44",fontsize=12)
ax.set_xticks([-30,lower,0,12,upper,50])
ax.set_xticklabels(["−30",f"{lower:.2f}\nQ2.5","0", "12\nmean",f"{upper:.2f}\nQ97.5","+50"])
ax.set(xlabel="Capacity difference Δ = B − A (Mt CO₂)",
       ylabel="Number of draws per bin",
       title="Figure 9 — Q2.5 and Q97.5 from 20,000 simulated differences")
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y",alpha=.12)
plt.show()

**Interpretation:** The interval is approximately **−11.43 to 35.29 Mt**. It includes differences favouring both sites. This is a model-based uncertainty range for possible *differences*, not a confidence interval for their mean.

### 4.3 Statistical quantiles vs storage-industry exceedance values

A **statistical quantile $Q_{10}$** has about 10% of distribution values below it. A **storage-industry P90 capacity** is a low value with about 90% of distribution values **above** it. Thus, for a continuous distribution:

$$P90=Q_{10},\qquad P50=Q_{50},\qquad P10=Q_{90}.$$

Here we illustrate this convention on **Site A's individual simulated capacity**, not on signed differences. Keep capital $Q$ separate from PROMETHEE lower-case $q$.

### Figure 10 — Statistical quantiles and exceedance probabilities

Three vertical lines mark *the same underlying values* using the two naming conventions.

In [ ]:
# @title Figure 10 — Statistical Q quantiles versus storage P exceedance
v10,v50,v90=np.quantile(a,[.10,.50,.90])
fig,ax=plt.subplots(figsize=(10.5,5.2))
fig.subplots_adjust(left=.09,right=.97,bottom=.2,top=.85)
site_bins=np.linspace(67,133,90)
hist,sites_edges=np.histogram(a,bins=site_bins,density=True)
site_centres=(sites_edges[:-1]+sites_edges[1:])/2
for value,xc,w in zip(hist,site_centres,np.diff(sites_edges)):
    c="#EFB8BD" if xc<v10 else ("#A9CCF4" if xc<=v90 else "#EFD5AC")
    ax.bar(xc,value,width=w,color=c,edgecolor="white",lw=.25)
for value,c in [(v10,RED),(v50,BLUE),(v90,ORANGE)]:
    ax.axvline(value,color=c,lw=1.8,ls="--")
ax.text(.13,.91,f"Q10 = P90\n{v10:.1f} Mt",transform=ax.transAxes,
        ha="center",va="top",color="#983540",weight="bold")
ax.text(.5,.91,f"Q50 = P50\n{v50:.1f} Mt",transform=ax.transAxes,
        ha="center",va="top",color="#154A90",weight="bold")
ax.text(.87,.91,f"Q90 = P10\n{v90:.1f} Mt",transform=ax.transAxes,
        ha="center",va="top",color="#945B16",weight="bold")
ax.set(xlim=(67,133),xlabel="Site A simulated capacity (Mt CO₂)",
       ylabel="Probability density",
       title="Figure 10 — Different names, identical percentile positions")
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y",alpha=.12)
plt.show()

**Interpretation:** The middle 80% of Site A's capacities lies between its statistical Q10/P90 and Q90/P10. These probabilistic capacity values are **not** PROMETHEE's preference thresholds.

---

## 5. From uncertainty to an exploratory candidate q — NOT a validated q

### 5.1 Which central probability interval should we use?

A statistical analyst can select a 50%, 68.3%, or 95% central uncertainty interval. **These percentages are coverage conventions; none is prescribed by PROMETHEE.**

We will compute the corresponding interval boundaries directly from our **same 20,000 simulated differences** (mean around 12 Mt), not from the separate five-draw example. This matters because we must not mix scenarios.

### Figure 11 — Central 50%, 68.3% and 95% intervals

Each panel uses the *same difference distribution*. Only the selected percentage changes. The half-width of each central interval provides an **uncertainty-width diagnostic**, not a justified decision q.

In [ ]:
# @title Figure 11 — Compare uncertainty coverage choices
interval_choices=[("50% central",.25,.75),
                  ("68.3% central",.158655,.841345),
                  ("95% central",.025,.975)]
fig,axes=plt.subplots(3,1,figsize=(10.5,10),sharex=True)
fig.subplots_adjust(left=.08,right=.97,top=.91,bottom=.09,hspace=.48)
xx=np.linspace(-40,65,1200)
sim_mu,sim_sd=difference.mean(),difference.std(ddof=1)
yy=normal_pdf(xx,sim_mu,sim_sd)
for ax,(name,lq,uq) in zip(axes,interval_choices):
    left,right=np.quantile(difference,[lq,uq])
    half=(right-left)/2
    ax.plot(xx,yy,color=BLUE,lw=2.5)
    ax.fill_between(xx,yy,where=(xx>=left)&(xx<=right),color="#A9CCF4",alpha=.88)
    ax.fill_between(xx,yy,where=(xx<left)|(xx>right),color="#EFB8BD",alpha=.7)
    for edge in (left,right):
        ax.axvline(edge,color=BLUE,ls="--",lw=1.2)
    ax.axvline(sim_mu,color=BLUE,ls=":",lw=1.3)
    ax.text(.98,.87,f"Half-width = {half:.2f} Mt",transform=ax.transAxes,
            ha="right",va="top",color=GREEN,weight="bold",fontsize=11,
            bbox=dict(facecolor="white",edgecolor="none",alpha=.90))
    ax.text(.50,.52,name,transform=ax.transAxes,color="#164A90",
            ha="center",fontsize=14,weight="bold")
    ax.set(xlim=(-38,60),ylim=(0,yy.max()*1.20),
           title=f"{name}: [{left:.2f}, {right:.2f}] Mt")
    ax.set_yticks([])
    ax.spines[["top","right","left"]].set_visible(False)
    ax.grid(axis="x",alpha=.1)
axes[-1].set_xlabel("Capacity difference Δ = B − A (Mt CO₂)",labelpad=10)
fig.suptitle("Figure 11 — Same Monte Carlo results, three coverage choices",
             fontsize=14,weight="bold")
plt.show()

**Interpretation:** The wider the central coverage, the larger the half-width. Our choice of 95% will therefore produce a larger candidate than a 50% choice. **Selecting 95% is an assumption to justify and test**, not a scientifically demonstrated definition of decision indifference.

### 5.2 Proposed exploratory q from the 95% interval

Our current learning rule is:

<details><summary>Show candidate-q half-width formula and its symbols</summary>

$$q^*_{AB}=\frac{Q_{97.5\%}(\Delta)-Q_{2.5\%}(\Delta)}{2}.$$

$Q_{2.5\%}$ and $Q_{97.5\%}$ are statistical quantiles of the **simulated difference**. The star marks an **exploratory candidate**, *not* a validated PROMETHEE threshold.

</details>

**Table 6 — Illustrative decision-rule parameters** reports q* and a provisional p set to 2q* solely to illustrate the PROMETHEE curve. **p is not calibrated in Task A.**

In [ ]:
# @title Table 6 — Exploratory q and illustrative p
q_candidate=float((upper-lower)/2)
p_illustration=float(max(2*q_candidate,q_candidate+1))
print("Table 6 — Illustrative decision-rule parameters; not scientifically calibrated")
display(pd.DataFrame({
    "Parameter":["Exploratory q* for B versus A","Illustrative p"],
    "Value (Mt)":[round(q_candidate,2),round(p_illustration,2)],
    "Interpretation":["Half-width of central 95% difference interval",
                      "Chosen as max(2q*, q*+1) solely for demonstration"]
}))

### Figure 12 — Where does the exploratory q* = 23.36 Mt come from?

The interval begins at approximately −11.43 Mt and ends at 35.29 Mt. Its full width is about 46.72 Mt. The exploratory q* is **half that width**. The **interval midpoint is not zero** and **is not the threshold**. The graphic measures width, not preference.

In [ ]:
# @title Figure 12 — Half-width of the central 95% difference interval
fig,ax=plt.subplots(figsize=(10.5,5.2))
fig.subplots_adjust(left=.09,right=.97,bottom=.19,top=.85)
ax.plot(xx,yy,color=BLUE,lw=2.7)
ax.fill_between(xx,yy,where=(xx>=lower)&(xx<=upper),color="#A9CCF4",alpha=.88)
ax.fill_between(xx,yy,where=(xx<lower)|(xx>upper),color="#EFB8BD",alpha=.75)
midpoint=float((lower+upper)/2)
for edge in (lower,upper):
    ax.axvline(edge,color=RED,ls="--",lw=1.5)
ax.axvline(midpoint,color=GREEN,ls=":",lw=1.7)
arrow_y=yy.max()*.93
ax.annotate("",xy=(upper,arrow_y),xytext=(midpoint,arrow_y),
            arrowprops=dict(arrowstyle="<->",color=GREEN,lw=2))
ax.text(.68,.96,f"Half-width q* = {q_candidate:.2f} Mt",
        transform=ax.transAxes,color=GREEN,ha="center",va="top",
        fontsize=12,weight="bold",
        bbox=dict(facecolor="white",edgecolor="none",alpha=.94))
ax.set(xlim=(-38,60),ylim=(0,yy.max()*1.3),
       xlabel="Simulated difference B − A (Mt CO₂)",
       ylabel="Normal-density approximation (for this fictional example)",
       title="Figure 12 — 95% interval width and exploratory q*")
ax.set_xticks([-30,lower,0,midpoint,upper,55])
ax.set_xticklabels(["−30",f"{lower:.2f}\nQ2.5","0",
                    f"{midpoint:.2f}\nmidpoint",f"{upper:.2f}\nQ97.5","55"],fontsize=9)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y",alpha=.1)
plt.show()

**Interpretation:** The number 23.36 Mt quantifies spread in our example. It is **not yet** scientifically justified as a difference below which storage sites should be considered indifferent.

### 5.3 Figure 13 — Statistical probability vs decision indifference

Two different questions must remain separate:

- **Monte Carlo:** How often does B have greater capacity than A? About 84% of our 20,000 draws.
- **PROMETHEE on mean capacities:** If we input the *mean difference*, which is about 12 Mt, and choose q* = 23.36 Mt, then the preference for B under this capacity criterion is **zero**.

This illustrates the consequences of the provisional rule. **Applying PROMETHEE to every simulated draw and then averaging preferences is a different approach** and may give a nonzero result even if the function applied to the mean gives zero.

In [ ]:
# @title Figure 13 — Probability and preference are different concepts
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11,4.6))
fig.subplots_adjust(left=.08,right=.97,bottom=.22,top=.83,wspace=.27)
ax1.hist(difference,bins=68,color="#A9CCF4",edgecolor="white",linewidth=.3)
ax1.axvline(0,color=RED,ls="--",lw=1.8)
ax1.axvline(difference.mean(),color=BLUE,lw=1.6)
ax1.text(.98,.95,f"P(B>A) = {p_b:.1%}",transform=ax1.transAxes,ha="right",va="top",
         color="#164A90",weight="bold",fontsize=11,
         bbox=dict(facecolor="white",edgecolor="none",alpha=.93))
ax1.set(xlabel="Simulated difference (Mt)",ylabel="Number of draws",
        title="A · Statistical comparison")
ax1.spines[["top","right"]].set_visible(False)
daxis=np.linspace(0,55,550)
pref=np.clip((daxis-q_candidate)/(p_illustration-q_candidate),0,1)
ax2.plot(daxis,pref,color=BLUE,lw=2.8)
ax2.axvline(q_candidate,color=GREEN,ls="--",lw=1.5)
ax2.plot([difference.mean()],[0],marker="o",color=RED,markersize=9)
ax2.annotate("Mean Δ ≈ 12 Mt\nPreference = 0",
             (difference.mean(),0),xytext=(15,.40),
             arrowprops=dict(arrowstyle="->",color=RED),fontsize=10,color="#98404A")
ax2.set(xlabel="Difference based on mean capacities (Mt)",
        ylabel="PROMETHEE preference degree",
        title="B · Decision rule applied to mean Δ",
        xlim=(0,55),ylim=(-.06,1.1))
ax2.spines[["top","right"]].set_visible(False)
ax2.grid(axis="y",alpha=.12)
plt.show()

**Interpretation:** 84% is a probability about *simulation draws*. A preference degree of zero is a *decision-rule output* for the mean difference. The apparent tension shows why we must validate a statistical rule before calling it indifference.

### 5.4 Figure 14 — PROMETHEE preference function with q* and illustrative p

For a **positive** capacity advantage $d=B-A$, our teaching model uses a linear preference with indifference:

<details><summary>Show PROMETHEE function and the 35 Mt example</summary>

$$P(B,A;d)=\begin{cases}0 & d\le q^*\\ \frac{d-q^*}{p-q^*} & q^*<d<p\\ 1 & d\ge p.\end{cases}$$

For $d=35$ Mt, q* ≈ 23.36 Mt, and p ≈ 46.72 Mt, the preference degree is $(35-23.36)/(46.72-23.36)\approx0.50$. This is **not a 50% probability**.

</details>

The shaded bands distinguish **no preference**, **partial preference**, and **full preference** for B *on capacity alone*.

In [ ]:
# @title Figure 14 — Annotated PROMETHEE preference function
xxp=np.linspace(0,70,600)
yp=np.clip((xxp-q_candidate)/(p_illustration-q_candidate),0,1)
fig,ax=plt.subplots(figsize=(10.5,5.4))
fig.subplots_adjust(left=.10,right=.97,bottom=.20,top=.85)
ax.axvspan(0,q_candidate,color="#DCEDE5",alpha=.8)
ax.axvspan(q_candidate,p_illustration,color="#FBEDCA",alpha=.85)
ax.axvspan(p_illustration,70,color="#DEE8FB",alpha=.85)
ax.plot(xxp,yp,color=BLUE,lw=3)
ax.axvline(q_candidate,color=GREEN,ls="--",lw=1.7)
ax.axvline(p_illustration,color=RED,ls="--",lw=1.7)
ax.text(q_candidate/2,.73,"INDIFFERENCE",color="#206B4E",
        fontsize=11,ha="center",weight="bold")
ax.text((p_illustration+q_candidate)/2,.76,"PARTIAL\nPREFERENCE",
        color="#8D691A",fontsize=11,ha="center",weight="bold")
ax.text((p_illustration+70)/2,.64,"FULL\nPREFERENCE",
        color="#1B559F",fontsize=11,ha="center",weight="bold")
for d,c in [(12,GREEN),(35,"#9A6419"),(50,BLUE)]:
    value=float(np.clip((d-q_candidate)/(p_illustration-q_candidate),0,1))
    ax.scatter([d],[value],s=80,color=c,zorder=4)
    ax.annotate(f"{d} Mt → {value:.2f}",xy=(d,value),
                xytext=(0,13 if d!=50 else -22),
                textcoords="offset points",ha="center",color=c,fontsize=10,weight="bold")
ax.set(xlim=(0,70),ylim=(-.04,1.08),
       xlabel="Positive capacity advantage of B relative to A (Mt CO₂)",
       ylabel="Single-criterion preference degree",
       title="Figure 14 — PROMETHEE preference with exploratory q* and illustrative p")
ax.set_xticks([0,12,q_candidate,35,p_illustration,50,70])
ax.set_xticklabels(["0","12",f"q*={q_candidate:.2f}","35",
                     f"p={p_illustration:.2f}","50","70"],fontsize=9)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y",alpha=.13)
plt.show()

**Interpretation:** A 12 Mt advantage is ignored by this provisional threshold; 35 Mt produces a partial preference around 0.50; and 50 Mt produces a full preference of 1. The **shape of the curve is a decision modelling choice**, not a probability distribution. The value p is introduced for illustration only; deriving p is **Task B**.

---

## 6. From Normal to beta-PERT — why it matters for storage capacity

The normal capacity distributions in Notebook A are for learning. In the GEUS-based notebook, geological inputs such as GRV, net-to-gross, porosity, CO₂ density and storage efficiency can instead use **beta-PERT**, when justified minimum, most likely and maximum values are available.

A beta-PERT distribution is bounded by its chosen minimum and maximum. A normal distribution has unbounded mathematical support. Neither choice is inherently correct for all geological quantities.

### Figure 15 — Normal versus beta-PERT for a fictional storage-efficiency input

Fictional input: minimum **5%**, most likely **10%**, maximum **20%**, beta-PERT shape parameter **4**. The normal curve is chosen to have the **same mean and SD** as the beta-PERT for a fair comparison of shape and tails. Both are **INPUT distributions**, not final storage-capacity distributions.

In [ ]:
# @title Figure 15 — Bounded PERT versus unbounded normal input
from math import lgamma
minimum,mode,maximum=5.0,10.0,20.0
lam=4.0
alpha=1+lam*(mode-minimum)/(maximum-minimum)
beta=1+lam*(maximum-mode)/(maximum-minimum)
pert_mean=minimum+(maximum-minimum)*alpha/(alpha+beta)
pert_sd=(maximum-minimum)*np.sqrt(alpha*beta/((alpha+beta)**2*(alpha+beta+1)))
xs=np.linspace(0,25,1200)
t=(xs-minimum)/(maximum-minimum)
pert_density=np.zeros_like(xs)
mask=(t>0)&(t<1)
pert_density[mask]=np.exp(
    (alpha-1)*np.log(t[mask])+(beta-1)*np.log1p(-t[mask])
    -(lgamma(alpha)+lgamma(beta)-lgamma(alpha+beta))
)/(maximum-minimum)
normal_density=normal_pdf(xs,pert_mean,pert_sd)
fig,ax=plt.subplots(figsize=(10.5,5.0))
fig.subplots_adjust(left=.09,right=.97,bottom=.19,top=.83)
ax.axvspan(0,minimum,color="#F1B5BA",alpha=.22)
ax.axvspan(maximum,25,color="#F1B5BA",alpha=.22)
ax.plot(xs,pert_density,color=BLUE,lw=2.7,label="Beta-PERT (bounded)")
ax.plot(xs,normal_density,color=ORANGE,lw=2.2,ls="--",
        label="Normal with same mean and SD (unbounded)")
ax.fill_between(xs,pert_density,color="#B9D6FA",alpha=.38)
for v,c in [(minimum,GREEN),(mode,BLUE),(maximum,GREEN)]:
    ax.axvline(v,color=c,lw=1.1,ls=":")
ax.set(xlim=(0,25),ylim=(0,max(pert_density.max(),normal_density.max())*1.23),
       xlabel="Fictional storage-efficiency input (%)",
       ylabel="Probability density",
       title="Figure 15 — Beta-PERT vs normal: different shapes and bounds")
ax.set_xticks([0,5,10,15,20,25])
ax.set_xticklabels(["0", "5\nminimum","10\nmost likely","15","20\nmaximum","25"])
ax.legend(loc="upper right",frameon=False,fontsize=9)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y",alpha=.11)
plt.show()

**Interpretation:** The beta-PERT has no probability below 5% or above 20% because of its **chosen model bounds**. The normal model has nonzero probability outside those limits. Choosing valid geological bounds requires evidence; simply choosing PERT does not guarantee the assumptions are scientifically correct.

**Very important:** multiplying several uncertain *input* parameters through the volumetric storage-capacity equation produces an *output capacity distribution*. That output does **not** automatically follow PERT or normal. Notebook B analyses the empirical Monte Carlo capacity arrays.

---

## 7. Transfer the same terminology to GEUS-based cases

**Table 7 — Crosswalk from fictional Notebook A to research Notebook B**

| Notebook A — fictional | Notebook B — GEUS-derived inputs | Common interpretation |
|---|---|---|
| Fictional Site A | Rødby–Bunter (default Site A) | Reference capacity $C_{A,i}$ |
| Fictional Site B | Havnsø–Gassum (default Site B) | Comparison capacity $C_{B,i}$ |
| Five invented pairs | New Monte Carlo results from published input assumptions | Difference $\Delta_i=C_{B,i}-C_{A,i}$ |
| Normal capacity inputs | Geological PERT input parameters | Site capacity distributions |
| Probability $P(B>A)$ | Pairwise empirical comparisons | Model-implied superiority probability |
| Difference Q2.5 and Q97.5 | Difference quantiles | Uncertainty interval |
| Exploratory q* and illustrative p | Research candidates to evaluate | **Not** calibrated PROMETHEE thresholds |

**Notebook B generates NEW Monte Carlo realisations from GEUS-derived parameters**, not the exact draws originally used by GEUS. It currently uses independent empirical resampling across sites. If sites share uncertainties, a carefully specified **joint uncertainty model** is needed.

### Figure 16 — How Task A fits into the wider CO2SITE workflow

We first model capacity, then compare uncertain capacities across sites, propose **candidate** q values, and finally examine whether the decision rule is defensible and robust. A complete PROMETHEE ranking also needs p, other criteria and weights.

In [ ]:
# @title Figure 16 — End-to-end CO2SITE workflow
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

labels=[
    ("Geological input distributions","PERT / other justified models"),
    ("Monte Carlo site capacities","Full simulated capacity arrays"),
    ("Pairwise difference distributions","Quantiles, SD, and P(B>A)"),
    ("Exploratory q candidates","Multiple documented candidate rules"),
    ("Validation and sensitivity analysis","Site comparability and ranking stability"),
    ("PROMETHEE ranking","Requires p, criterion weights, and safeguards")
]
fig,ax=plt.subplots(figsize=(9,8.0))
fig.subplots_adjust(left=.06,right=.94,top=.92,bottom=.08)
ax.set_xlim(0,10);ax.set_ylim(0,8.3);ax.axis("off")
for j,(name,detail) in enumerate(labels):
    y=7.25-j*1.27
    edge=GREEN if j==3 else "#7A98C3"
    face="#E7F2ED" if j==3 else "#EDF3FA"
    rect=FancyBboxPatch((.55,y),8.9,.75,
                        boxstyle="round,pad=0.12,rounding_size=0.12",
                        edgecolor=edge,linewidth=1.5,facecolor=face)
    ax.add_patch(rect)
    ax.text(5,y+.48,name,fontsize=12,weight="bold",ha="center",va="center",
            color="#1B3C67")
    ax.text(5,y+.18,detail,fontsize=9.5,ha="center",va="center",
            color="#42607F")
    if j<len(labels)-1:
        arrow=FancyArrowPatch((5,y-.13),(5,y-.39),
                              arrowstyle="-|>",mutation_scale=15,lw=1.4,
                              color="#6484AF")
        ax.add_patch(arrow)
ax.set_title("Figure 16 — From geological uncertainty to PROMETHEE decisions",
             fontsize=14,weight="bold",pad=18)
plt.show()

**Interpretation:** This is a research workflow, **not** evidence that q has already been established. A defensible common q for many storage sites must be transparent, scientifically motivated and robust in sensitivity analysis.

**Task A conclusion:** we have learned how to compute capacity differences and uncertainty diagnostics, including one *exploratory* q*. We have **not** demonstrated that a statistical SD, percentile width or superiority probability uniquely defines the decision-maker's indifference threshold.

**Next: apply the same definitions to GEUS-derived input assumptions.**

[Open Notebook B — GEUS-based case studies](https://colab.research.google.com/github/AnaSoles/ggg-co2-storage-eval/blob/research/task-a-capacity-indifference/examples/colab_task_a_capacity_q.ipynb)

**Research objective reminder:** How to derive scientifically justified indifference (q) and preference (p) thresholds for CO₂ storage capacity in PROMETHEE. A reproducible procedure must be evaluated across multiple CO₂ storage sites for CO2SITE.